[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_plots.ipynb)

# Figures for the screening funnel

**Blue group · Cryptosporidiosis**

This notebook collects the figures for the group's talk and paper in one place. The first one
puts the whole screening funnel on a single map, so you can see at a glance how 28,732
molecules became 1,000, and where silymarin sits among them.

## What you will do

- Load the three stages of the funnel, with the map coordinates already computed for them
- Rebuild the thousand least toxic molecules, using the same rule as the filter notebook
- Draw all of them, and silymarin, on one t-SNE map

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "blue"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. The four sets of molecules

The group's screen narrowed a large library down in three steps, and each set is contained in
the one before it:

| Molecules | What they are | Notebook |
|---|---|---|
| 28,732 | Hits of the Pharmit pharmacophore screen of the MolPort library | `blue_pharmit_hits` |
| 1,887 | Those whose 3D shape resembles silymarin's | `blue_sand_shape_similarity` |
| 1,000 | Those that also look least toxic to human cells | `blue_cytotoxicity_filter` |

To draw them we need a position for every molecule on a map. We do not calculate one here.
The Ersilia model [eos1klk](https://github.com/ersilia-os/eos1klk) has already placed all
28,732 molecules on a map built from 1.3 million reference compounds, and its output is in
`data/`. Using the same precomputed map everywhere means a molecule sits in the same place in
every notebook in this project, which it would not if each notebook fitted its own.

In [ ]:
from scripts import figures, chemspace

space, hits, kept, seed = figures.load_funnel()

print(f"{len(space):,} Pharmit hits")
print(f"{len(hits):,} of them match silymarin's shape")
print(f"{len(kept):,} of those survive the toxicity filter")

Each row holds one molecule: its catalogue number, its structure as a SMILES string, its
position on the map, and the toxicity the model predicted for it.

In [ ]:
hits[["molport_id", "smiles", "tsne_x", "tsne_y", figures.TOXICITY]].head()

## 2. Rebuilding the thousand

`blue_cytotoxicity_filter` saved its shortlist into `outputs/`, but that folder is deliberately
kept out of the repository, so it does not exist when this notebook runs in Colab. Rather than
depend on a file that may not be there, we rebuild the shortlist from the data that *is*
committed, using that notebook's rule exactly: sort the 1,887 shape hits by their predicted
toxicity to liver cells, and keep the thousand lowest.

The numbers run from 0 to 1, and **higher means more likely to be toxic** — the opposite
direction to most of the other scores in this project, which is easy to get backwards.

In [ ]:
cutoff = kept[figures.TOXICITY].max()
dropped = hits[~hits["smiles"].isin(kept["smiles"])]

print(f"kept {len(kept):,} molecules scoring {cutoff:.3f} and below")
print(f"median toxicity, kept:    {kept[figures.TOXICITY].median():.3f}")
print(f"median toxicity, dropped: {dropped[figures.TOXICITY].median():.3f}")

> **Note:** The filter ranks on liver cells alone. Liver and muscle agree almost completely,
> but lung agrees much less, so this choice does throw some information away.
> `blue_cytotoxicity_filter` looks at that in detail.

## 3. The funnel on one map

Now everything goes on one picture. The sets are nested, so they are drawn largest first and
smallest last, which makes the nesting readable:

- a faint grey dot is one of the 28,732 Pharmit hits
- a large turquoise dot is a shape hit that the toxicity filter **dropped**
- a turquoise dot with a pink centre is a shape hit that was **kept**
- the star is silymarin, the molecule the whole pharmacophore was built from

In [ ]:
import stylia

stylia.set_format("slide")
stylia.set_style("article")  # the NPG palette: turquoise, fuchsia, amber, silver ...

figure, axes = stylia.create_figure(1, 1, width=0.6, height=1.0)
ax = axes.next()
figures.plot_funnel(ax, space, hits, kept, seed, projection="tsne")
figures.add_legend(ax)
chemspace.label_space(ax, "tsne",
                      f"From {len(space):,} Pharmit hits to {len(kept):,} candidates")
figure.tight_layout()

The shape hits are spread right across the map rather than sitting in one corner, and the
toxicity filter thins them out fairly evenly. So the funnel kept a wide range of chemistry
instead of collapsing onto one family of molecules, which is what we want from it.

> **Note:** On a t-SNE map, which molecules are *near each other* is meaningful, but the
> distance between two far-apart points is not, and the size of an empty gap means nothing at
> all. Read it as neighbourhoods, not as a ruler.

> **Exercise:** `eos1klk` also returns UMAP, PCA and TMAP coordinates. Change `projection="tsne"`
> above to `"umap"` and run the cell again. Do the same groups of molecules stay together?

## Summary

- The screening funnel went from 28,732 Pharmit hits, to 1,887 that match silymarin's shape,
  to the 1,000 of those that look least toxic.
- Drawn on one t-SNE map, the survivors are spread across the whole space, so the filters kept
  a varied set rather than one family of molecules.
- Silymarin sits inside a well-populated region, so the screen did find molecules near the
  compound it started from.

**Next:** more figures belong in this notebook as the project grows — the distribution of shape
similarity scores, the cytotoxicity predictions, and the SPRINT ranking. Add each one as a new
numbered section.